# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UmairAsim180/1_FlyRank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule (plain words): for each page, compare its actual CTR to the average CTR other
pages get at the same search position (the "position benchmark"). If a page underperforms
that benchmark AND has real impression volume, it's leaking clicks it should be getting —
flag it for a CTR fix, scored by how many clicks it's actually losing (impact-weighted).

Reason code: CTR_BELOW_POSITION_BENCHMARK
Action label: fix_ctr
Score: (position_benchmark_ctr - actual_ctr) * impressions   [estimated clicks lost]

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip install -q duckdb
import duckdb
from google.colab import userdata

token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{token}')")
REL = "hf://datasets/FlyRank/internship-warehouse"

signal1 = con.sql(f"""
    WITH page_month AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            AVG(gsc_avg_position) AS avg_position
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE client_has_gsc IS TRUE AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) > 0
    )
    SELECT
        CASE
            WHEN avg_position <= 3 THEN '1-3'
            WHEN avg_position <= 10 THEN '4-10'
            WHEN avg_position <= 20 THEN '11-20'
            WHEN avg_position <= 50 THEN '21-50'
            ELSE '51+'
        END AS position_bucket,
        COUNT(*) AS n,
        AVG(clicks * 1.0 / impressions) AS avg_ctr
    FROM page_month
    GROUP BY position_bucket
    ORDER BY MIN(avg_position)
""").df()
print(signal1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  position_bucket      n   avg_ctr
0             1-3  17578  0.012399
1            4-10  81988  0.004926
2           11-20  32203  0.003211
3           21-50  33288  0.002287
4             51+  11681  0.000903


Verdict: CONFIRMED / OPPOSITE / MIXED / FALSE — [state which, based on whether avg_ctr
drops as position_bucket worsens]. [One sentence on what the numbers actually show.]

In [8]:
signal2 = con.sql(f"""
    WITH page_month AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            AVG(gsc_avg_position) AS avg_position
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE client_has_gsc IS TRUE AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) > 0
    )
    SELECT
        CASE
            WHEN impressions < 50 THEN 'under_50'
            WHEN impressions < 200 THEN '50-200'
            WHEN impressions < 1000 THEN '200-1000'
            ELSE '1000+'
        END AS impression_bucket,
        COUNT(*) AS n,
        AVG(clicks * 1.0 / impressions) AS avg_ctr,
        AVG(avg_position) AS avg_position
    FROM page_month
    GROUP BY impression_bucket
    ORDER BY MIN(impressions)
""").df()
print(signal2)

  impression_bucket      n   avg_ctr  avg_position
0          under_50  60624  0.008380     16.957822
1            50-200  31281  0.002455     22.094926
2          200-1000  39775  0.002368     15.387970
3             1000+  45058  0.002950     11.017385


Verdict: [CONFIRMED / OPPOSITE / MIXED / FALSE] — [does higher-impression volume
correlate with the kind of pattern quick-win logic assumes — good position, low CTR,
real opportunity? Or not?]

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
scored = con.sql(f"""
    WITH page_month AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            AVG(gsc_avg_position) AS avg_position,
            SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS actual_ctr
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE client_has_gsc IS TRUE AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) > 0
    ),
    position_benchmark AS (
        SELECT
            CASE
                WHEN avg_position <= 3 THEN '1-3'
                WHEN avg_position <= 10 THEN '4-10'
                WHEN avg_position <= 20 THEN '11-20'
                WHEN avg_position <= 50 THEN '21-50'
                ELSE '51+'
            END AS position_bucket,
            AVG(actual_ctr) AS benchmark_ctr
        FROM page_month
        GROUP BY position_bucket
    )
    SELECT
        p.content_hash_id,
        p.impressions,
        p.clicks,
        p.avg_position,
        p.actual_ctr,
        b.benchmark_ctr,
        (b.benchmark_ctr - p.actual_ctr) * p.impressions AS score,
        'CTR_BELOW_POSITION_BENCHMARK' AS reason_code,
        'fix_ctr' AS action
    FROM page_month p
    JOIN position_benchmark b
        ON b.position_bucket = CASE
            WHEN p.avg_position <= 3 THEN '1-3'
            WHEN p.avg_position <= 10 THEN '4-10'
            WHEN p.avg_position <= 20 THEN '11-20'
            WHEN p.avg_position <= 50 THEN '21-50'
            ELSE '51+'
        END
    ORDER BY score DESC
""").df()

import os
os.makedirs('work/outputs', exist_ok=True)
scored.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"Wrote {len(scored)} rows.")
scored.head(20)

Wrote 176738 rows.


,content_hash_id,impressions,clicks,avg_position,actual_ctr,benchmark_ctr,score,reason_code,action
0,content_8d7d99f109e19aa2,203497.0,289.0,2.563756,0.001420,0.012399,2234.248399,CTR_BELOW_POSITION_BENCHMARK,fix_ctr
1,content_0e03de7680314cd5,221310.0,720.0,2.675217,0.003253,0.012399,2024.119585,CTR_BELOW_POSITION_BENCHMARK,fix_ctr
2,content_eadb33b5df496f4a,617124.0,5668.0,2.383011,0.009185,0.012399,1983.990668,CTR_BELOW_POSITION_BENCHMARK,fix_ctr
3,content_4ffe18112a5642e3,186983.0,586.0,2.331060,0.003134,0.012399,1732.484083,CTR_BELOW_POSITION_BENCHMARK,fix_ctr
4,content_ec2e0346994fb5a5,245276.0,1480.0,2.854514,0.006034,0.012399,1561.284512,CTR_BELOW_POSITION_BENCHMARK,fix_ctr
5,content_545bb6cc7081ded3,122905.0,287.0,2.615390,0.002335,0.012399,1236.952906,CTR_BELOW_POSITION_BENCHMARK,fix_ctr
6,content_44f34c0a90047651,212404.0,24.0,7.346909,0.000113,0.004926,1022.313231,CTR_BELOW_POSITION_BENCHMARK,fix_ctr
7,content_9ef3d7516483e665,89229.0,92.0,2.481596,0.001031,0.012399,1014.389438,CTR_BELOW_POSITION_BENCHMARK,fix_ctr
8,content_306bc78dff1eb683,80821.0,35.0,1.488604,0.000433,0.012399,967.134964,CTR_BELOW_POSITION_BENCHMARK,fix_ctr
9,content_987d251ee617d9c6,152806.0,940.0,2.823429,0.006152,0.012399,954.708496,CTR_BELOW_POSITION_BENCHMARK,fix_ctr


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = scored.head(20)
top20[['content_hash_id', 'impressions', 'avg_position', 'actual_ctr', 'benchmark_ctr', 'score', 'action']]


,content_hash_id,impressions,avg_position,actual_ctr,benchmark_ctr,score,action
0,content_8d7d99f109e19aa2,203497.0,2.563756,0.001420,0.012399,2234.248399,fix_ctr
1,content_0e03de7680314cd5,221310.0,2.675217,0.003253,0.012399,2024.119585,fix_ctr
2,content_eadb33b5df496f4a,617124.0,2.383011,0.009185,0.012399,1983.990668,fix_ctr
3,content_4ffe18112a5642e3,186983.0,2.331060,0.003134,0.012399,1732.484083,fix_ctr
4,content_ec2e0346994fb5a5,245276.0,2.854514,0.006034,0.012399,1561.284512,fix_ctr
5,content_545bb6cc7081ded3,122905.0,2.615390,0.002335,0.012399,1236.952906,fix_ctr
6,content_44f34c0a90047651,212404.0,7.346909,0.000113,0.004926,1022.313231,fix_ctr
7,content_9ef3d7516483e665,89229.0,2.481596,0.001031,0.012399,1014.389438,fix_ctr
8,content_306bc78dff1eb683,80821.0,1.488604,0.000433,0.012399,967.134964,fix_ctr
9,content_987d251ee617d9c6,152806.0,2.823429,0.006152,0.012399,954.708496,fix_ctr


1. content_hash_id [xxxx]: fix_ctr — position ~[X], actual CTR far below the [bucket]
   benchmark, [impressions] impressions means the click loss is real, not noise. Would
   be wrong if: this page's true topic/intent doesn't match what people expect at this
   position (e.g. a listicle ranking for a transactional query) — low CTR would then be
   correct behavior, not a fixable problem.
... [repeat for rows 2–20, changing the specific numbers and the "would be wrong if" reasoning per row]

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: [name 1-2 rows from the top 20 that look shakiest — e.g. very low impressions
even within their bucket, meaning the "lost clicks" number is more noise than signal] —
score can look large even at low volume because CTR ratios are unstable with small
denominators.

Leakage check: this rule only uses impressions, clicks, and position — all fully observed
within the same month, no future window, no product/GA4 flags, no label-derived columns.
The benchmark itself is computed from the same month's data, not from any outcome after
the decision point.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(scored.head(20)[['impressions']].describe())

         impressions
count      20.000000
mean   142202.100000
std    127977.660748
min     60172.000000
25%     69709.000000
50%     83570.000000
75%    191111.500000
max    617124.000000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.